In [5]:

import os, sys, tempfile
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import mlflow
import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler

#### 1. Environment variable

In [6]:
MLFLOW_TRACKING_URI = "http://145.38.192.114:5002"   # <-- paste your server URL here
EXPERIMENT_NAME     = "team_transformer_magic"

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
BASE_DIR  = Path("/content") if IN_COLAB else Path(".")
DATA_DIR  = (BASE_DIR / "data").resolve()  # absolute paths matter for Ray
RAY_DIR   = str((BASE_DIR / "ray_results").resolve())

device = "cuda" if torch.cuda.is_available() else "cpu"


##### 1.1 Data set

In [7]:
# Pre-download MNIST ONCE before Ray spawns workers (otherwise every trial re-downloads)
datasets.MNIST(DATA_DIR, train=True,  download=True)
datasets.MNIST(DATA_DIR, train=False, download=True)

100.0%
100.0%
100.0%
100.0%


Dataset MNIST
    Number of datapoints: 10000
    Root location: /Users/saifzaman/Documents/DataScience/MADS-Transformer/data
    Split: Test

#### 2. Model parameterised by the search config

In [8]:

class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim, hidden):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, hidden, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(hidden, hidden * 2, 3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(hidden * 2 * 7 * 7, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden * 2 * 7 * 7),
            nn.Unflatten(1, (hidden * 2, 7, 7)),
            nn.ConvTranspose2d(hidden * 2, hidden, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(hidden, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z




#### 3. Trainable — Ray runs this once per config 

In [9]:

def train_autoencoder(config):
    # Each Ray trial is a fresh process, so re-connect to MLflow inside the trial.
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(EXPERIMENT_NAME)

    with mlflow.start_run():
        mlflow.log_params(config)
        mlflow.set_tag("framework", "pytorch")
        mlflow.set_tag("search",   "ray-tune-transformer")

        # Data
        tfm = transforms.ToTensor()
        full_train = datasets.MNIST(DATA_DIR, train=True, transform=tfm)
        train_ds, val_ds = random_split(
            full_train, [55_000, 5_000],
            generator=torch.Generator().manual_seed(42),
        )
        train_dl = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)
        val_dl   = DataLoader(val_ds,   batch_size=256)

        # Model
        model     = ConvAutoencoder(config["latent_dim"], config["hidden"]).to(device)
        optimizer = optim.Adam(model.parameters(), lr=config["lr"])
        loss_fn   = nn.MSELoss()

        for epoch in range(config["epochs"]):
            # --- train ---
            model.train()
            tot, n = 0.0, 0
            for x, _ in train_dl:
                x = x.to(device)
                x_hat, _ = model(x)
                loss = loss_fn(x_hat, x)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                tot += loss.item() * x.size(0); n += x.size(0)
            train_loss = tot / n

            # --- validate ---
            model.eval()
            tot, n = 0.0, 0
            with torch.no_grad():
                for x, _ in val_dl:
                    x = x.to(device)
                    x_hat, _ = model(x)
                    tot += loss_fn(x_hat, x).item() * x.size(0); n += x.size(0)
            val_loss = tot / n

            # log to MLflow (per-epoch curves)
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_loss",   val_loss,   step=epoch)

            # report to Ray — ASHA uses this to decide whether to kill this trial
            tune.report({"val_loss": val_loss, "train_loss": train_loss, "epoch": epoch})

        # Save final model as an MLflow artifact (lands on the remote server)
        with tempfile.TemporaryDirectory() as td:
            ckpt = Path(td) / "autoencoder.pt"
            torch.save(model.state_dict(), ckpt)
            mlflow.log_artifact(str(ckpt), artifact_path="model")



####  4. Search space

In [10]:

search_space = {
    "lr":         tune.loguniform(1e-4, 1e-2),
    "latent_dim": tune.choice([16, 32, 64, 128]),
    "hidden":     tune.choice([8, 16, 32]),
    "batch_size": tune.choice([64, 128, 256]),
    "epochs":     8,    # fixed cap; ASHA prunes losers early
}




#### 5. Launch the search

In [11]:

ray.init(ignore_reinit_error=True)

scheduler = ASHAScheduler(
    max_t=8,           # max epochs a trial can run
    grace_period=2,    # give every trial at least 2 epochs before it's allowed to be killed
    reduction_factor=3 # at each rung, keep top 1/3 of trials
)

# How much hardware each trial takes. On Colab's single GPU, gpu=0.5 lets 2 run in parallel.
# If you hit OOM, raise it to 1.0 so trials run sequentially.
resources = {"cpu": 2, "gpu": 0.5 if torch.cuda.is_available() else 0}

tuner = tune.Tuner(
    tune.with_resources(train_autoencoder, resources),
    param_space=search_space,
    tune_config=tune.TuneConfig(
        metric="val_loss",
        mode="min",
        num_samples=12,        # try 12 random configs
        scheduler=scheduler,
    ),
    run_config=tune.RunConfig(
        name="ae_search",
        storage_path=RAY_DIR,
    ),
)
results = tuner.fit()

best = results.get_best_result(metric="val_loss", mode="min")
print("Best config :", best.config)
print("Best val_loss:", best.metrics["val_loss"])

(train_autoencoder pid=18002) 2026/06/08 06:49:50 INFO mlflow.tracking.fluent: Experiment with name 'team_transformer_magic' does not exist. Creating a new experiment.


(train_autoencoder pid=18000) 🏃 View run luxuriant-shrimp-217 at: http://145.38.192.114:5002/#/experiments/1/runs/796d472c435a45c6a01dac0173fc2f0c
(train_autoencoder pid=18000) 🧪 View experiment at: http://145.38.192.114:5002/#/experiments/1
(train_autoencoder pid=18113) 🏃 View run suave-squirrel-891 at: http://145.38.192.114:5002/#/experiments/1/runs/c7c8a6c136af4b7692608ffceaec253c [repeated 2x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(train_autoencoder pid=18113) 🧪 View experiment at: http://145.38.192.114:5002/#/experiments/1 [repeated 2x across cluster]


2026-06-08 06:52:00,180	WARNING tune.py:219 -- Stop signal received (e.g. via SIGINT/Ctrl+C), ending Ray Tune run. This will try to checkpoint the experiment state one last time. Press CTRL+C (or send SIGINT/SIGKILL/SIGTERM) to skip. 
2026-06-08 06:52:00,187	INFO tune.py:1001 -- Wrote the latest version of all result files and experiment state to '/Users/saifzaman/Documents/DataScience/MADS-Transformer/ray_results/ae_search' in 0.0057s.


(train_autoencoder pid=17999) 🏃 View run industrious-cow-911 at: http://145.38.192.114:5002/#/experiments/0/runs/a44fb3a7417246fb9c1701918e449d65
(train_autoencoder pid=17999) 🧪 View experiment at: http://145.38.192.114:5002/#/experiments/0
(train_autoencoder pid=18112) 🏃 View run youthful-smelt-846 at: http://145.38.192.114:5002/#/experiments/1/runs/399aa2ae243c49909654a36bdf87207f
(train_autoencoder pid=18112) 🧪 View experiment at: http://145.38.192.114:5002/#/experiments/1


2026-06-08 06:52:10,309	INFO tune.py:1033 -- Total run time: 145.36 seconds (135.20 seconds for the tuning loop).
2026-06-08 06:52:10,311	WARNING tune.py:1048 -- Experiment has been interrupted, but the most recent state was saved.
Resume experiment with: Tuner.restore(path="/Users/saifzaman/Documents/DataScience/MADS-Transformer/ray_results/ae_search", trainable=...)
2026-06-08 06:52:10,327	WARNING experiment_analysis.py:180 -- Failed to fetch metrics for 5 trial(s):
- train_autoencoder_78454_00007: FileNotFoundError('Could not fetch metrics for train_autoencoder_78454_00007: both result.json and progress.csv were not found at /Users/saifzaman/Documents/DataScience/MADS-Transformer/ray_results/ae_search/train_autoencoder_78454_00007_7_batch_size=128,hidden=8,latent_dim=32,lr=0.0002_2026-06-08_06-49-45')
- train_autoencoder_78454_00008: FileNotFoundError('Could not fetch metrics for train_autoencoder_78454_00008: both result.json and progress.csv were not found at /Users/saifzaman/Docu

Best config : {'lr': 0.0049150165259106565, 'latent_dim': 32, 'hidden': 16, 'batch_size': 64, 'epochs': 8}
Best val_loss: 0.005761784598231316
